In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

%store -r full_pop_train
%store -r full_pop_test
%store -r claim_train
%store -r claim_test
%store -r full_pop_df

In [2]:
freq_model = smf.glm(
    formula="ClaimNb ~ DrivAge + BonusMalus + Density + C(Region) + VehBrand",
    data=full_pop_train,
    family=sm.families.Poisson(),
    offset=np.log(full_pop_train["Exposure"]),
).fit()

In [3]:
print(freq_model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                ClaimNb   No. Observations:               542410
Model:                            GLM   Df Residuals:                   542375
Model Family:                 Poisson   Df Model:                           34
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:            -1.1505e+05
Date:                Sat, 15 Aug 2026   Deviance:                   1.7466e+05
Time:                        11:54:12   Pearson chi2:                 1.45e+06
No. Iterations:                     7   Pseudo R-squ. (CS):           0.008401
Covariance Type:            nonrobust                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept           -3.9618      0.044  

In [4]:
sev_model = smf.glm(
    formula="ClaimTotalCapped ~ DrivAge + C(Region) + C(VehBrand) + VehPower",
    data=claim_train,
    family=sm.families.Gamma(link=sm.families.links.Log()),
).fit()

In [5]:
print(sev_model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:       ClaimTotalCapped   No. Observations:                19911
Model:                            GLM   Df Residuals:                    19877
Model Family:                   Gamma   Df Model:                           33
Link Function:                    Log   Scale:                          2.7399
Method:                          IRLS   Log-Likelihood:            -1.7339e+05
Date:                Sat, 15 Aug 2026   Deviance:                       22420.
Time:                        11:54:12   Pearson chi2:                 5.45e+04
No. Iterations:                    12   Pseudo R-squ. (CS):           0.003597
Covariance Type:            nonrobust                                         
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              7.3689      0

In [6]:
full_pop_test["pred_freq"] = freq_model.predict(full_pop_test, offset=np.log(full_pop_test["Exposure"]))
full_pop_test["pred_sev"] = sev_model.predict(full_pop_test)
full_pop_test["pred_rate"] = full_pop_test["pred_freq"] / full_pop_test["Exposure"]
full_pop_test["decile"] = pd.qcut(full_pop_test["pred_rate"], 10, labels=False)

freq_validation = full_pop_test.groupby("decile").apply(
    lambda g: pd.Series({
        "actual_freq": g["ClaimNb"].sum() / g["Exposure"].sum(),
        "predicted_freq": g["pred_freq"].sum() / g["Exposure"].sum(),
        "n_policies": len(g)
    })
)
print(freq_validation)

        actual_freq  predicted_freq  n_policies
decile                                         
0          0.053269        0.065145     13562.0
1          0.066990        0.072086     13559.0
2          0.071034        0.076389     13560.0
3          0.080329        0.081063     13560.0
4          0.085066        0.086854     13561.0
5          0.102500        0.095788     13560.0
6          0.122480        0.107724     13560.0
7          0.128212        0.120635     13560.0
8          0.145295        0.144134     13561.0
9          0.227999        0.227754     13560.0


In [7]:
claim_test["pred_sev"] = sev_model.predict(claim_test)
claim_test["decile"] = pd.qcut(claim_test["pred_sev"], 10, labels=False)

sev_validation = claim_test.groupby("decile").agg(
    actual_sev=("ClaimTotalCapped", "mean"),
    predicted_sev=("pred_sev", "mean"),
    n_claims=("ClaimTotalCapped", "count")
)
print(sev_validation)

         actual_sev  predicted_sev  n_claims
decile                                      
0       1749.081766    1491.975570       504
1       1752.646859    1577.475212       503
2       1557.004742    1606.162286       504
3       1550.516733    1630.633410       502
4       1757.885655    1654.902703       504
5       1508.151849    1685.329984       503
6       2036.953459    1722.307291       503
7       2029.570775    1793.026830       503
8       2138.828708    1904.923731       503
9       2187.625734    2126.195246       504


In [8]:
full_pop_df["pred_freq"] = freq_model.predict(full_pop_df, offset=np.log(full_pop_df["Exposure"]))
full_pop_df["pred_sev"] = sev_model.predict(full_pop_df)
full_pop_df["pred_rate"] = full_pop_df["pred_freq"] / full_pop_df["Exposure"]

In [9]:
%store freq_validation
%store sev_validation
%store full_pop_test
%store claim_test
%store full_pop_df

Stored 'freq_validation' (DataFrame)
Stored 'sev_validation' (DataFrame)
Stored 'full_pop_test' (DataFrame)
Stored 'claim_test' (DataFrame)
Stored 'full_pop_df' (DataFrame)
